In [ ]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_json(PROJECT_ROOT / "datasets" / "ViNumQA" / "test.json")
df.sample(n=5)

In [ ]:
len(df)

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
df["input_question"] = df.apply(lambda x: processing_input_question(x), axis=1)
df["program_processed"] = df.apply(lambda x: processing_program_content(x), axis=1)
df["answer_processed"] = df.apply(lambda x: processing_answer_content(x), axis=1)
df.sample(n=5)

In [ ]:
df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
# Flat column list, not [[...]] -- double brackets create a MultiIndex that
# silently breaks boolean filtering on df["generated_program"] later.
df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
df["generated_program"] = ""
df["calculated_program"] = ""
df

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}
 
[TABLE]
{table}
 
[TEXT AFTER TABLE]
{post_text}
 
### QUESTION:
{question}
 
### PROGRAM:"""

## Batch API generation (0-shot, gpt-5-nano)

Uses OpenAI's Batch API (`/v1/responses`) instead of a synchronous per-row loop:
50% lower cost than real-time calls, in exchange for async turnaround (usually
minutes, up to 24h). This is a good fit here since we're scoring the full
497-sample test set offline rather than serving live requests.

Reasoning effort is `medium`: with no in-context examples, the model needs more
deliberation to get the operator syntax and table grounding right (0-shot has
nothing to imitate). See the few-shot batch notebook for the `low`-effort variant.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Requires a real OpenAI API key (not the FPT Cloud / third-party BASE_URL used
# by the other notebooks) -- Batch API is an OpenAI-account-level feature.
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-5-nano"
REASONING_EFFORT = "medium"  # 0-shot: no examples to imitate, needs more deliberation

In [ ]:
import json
from pathlib import Path

BATCH_DIR = Path(f"outputs/0shot_{MODEL}_batch")
BATCH_DIR.mkdir(parents=True, exist_ok=True)

REQUESTS_PATH = BATCH_DIR / "requests.jsonl"

def build_request(df_index, values):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=values["pre_text"],
        table=values["table"],
        post_text=values["post_text"],
        question=values["question"],
    )
    return {
        "custom_id": str(df_index),
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODEL,
            "instructions": SYSTEM_MESSAGE,
            "input": user_msg,
            "reasoning": {"effort": REASONING_EFFORT},
            "max_output_tokens": 2048,
        },
    }

with open(REQUESTS_PATH, "w", encoding="utf-8") as f:
    for df_index, values in df.iterrows():
        f.write(json.dumps(build_request(df_index, values), ensure_ascii=False) + "\n")

print(f"Wrote {len(df)} requests to {REQUESTS_PATH}")

In [ ]:
# Submit the batch job. Re-run-safe: if a batch_id was already saved from a
# previous run of this notebook, reuse it instead of submitting a duplicate job.
BATCH_ID_PATH = BATCH_DIR / "batch_id.txt"

if BATCH_ID_PATH.exists():
    batch_id = BATCH_ID_PATH.read_text().strip()
    print(f"Reusing existing batch job: {batch_id}")
else:
    batch_input_file = client.files.create(
        file=open(REQUESTS_PATH, "rb"),
        purpose="batch",
    )
    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={"description": f"ViNumQA 0-shot {MODEL} ({REASONING_EFFORT} effort)"},
    )
    batch_id = batch.id
    BATCH_ID_PATH.write_text(batch_id)
    print(f"Submitted batch job: {batch_id}")

In [ ]:
# Poll until the batch completes. Safe to re-run this cell repeatedly --
# it just checks status and returns immediately if not done yet.
import time

TERMINAL_STATUSES = {"completed", "failed", "expired", "cancelled"}

def poll_batch(batch_id, poll_interval_s=30, max_wait_s=None):
    waited = 0
    while True:
        batch = client.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f"status={batch.status}  completed={counts.completed}/{counts.total}  failed={counts.failed}")
        if batch.status in TERMINAL_STATUSES:
            return batch
        if max_wait_s is not None and waited >= max_wait_s:
            print(f"Stopped polling after {waited}s (still {batch.status}). Re-run this cell later to keep waiting.")
            return batch
        time.sleep(poll_interval_s)
        waited += poll_interval_s

# max_wait_s=None polls indefinitely; set e.g. max_wait_s=600 to check in, then
# stop and resume later (batch jobs can take minutes to ~24h).
batch = poll_batch(batch_id, poll_interval_s=30, max_wait_s=None)

In [ ]:
# Download and parse the batch output, mapping results back onto df via custom_id.
if batch.status != "completed":
    raise RuntimeError(f"Batch did not complete successfully: status={batch.status}, errors={batch.errors}")

output_file = client.files.content(batch.output_file_id)
OUTPUT_PATH = BATCH_DIR / "results.jsonl"
output_file.write_to_file(OUTPUT_PATH)

n_ok, n_err = 0, 0
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        df_index = int(record["custom_id"])
        response = record.get("response")
        if response is None or response.get("status_code") != 200:
            n_err += 1
            continue
        body = response["body"]
        output_text = body.get("output_text")
        if output_text is None:
            # Fall back to walking the output items if output_text is absent.
            texts = []
            for item in body.get("output", []):
                for content in item.get("content", []):
                    if content.get("type") == "output_text":
                        texts.append(content["text"])
            output_text = "".join(texts)
        df.at[df_index, "generated_program"] = output_text.strip().strip("\n")
        n_ok += 1

print(f"Parsed {n_ok} successful / {n_err} failed responses out of {n_ok + n_err}.")

In [ ]:
import sys
sys.path.insert(0, "../../evaluate")  # fallback: relative path when running locally

from scorer import evaluate_dataframe  # noqa: E402

# scorer.py is the shared ViNumQA evaluator (notebooks/evaluate/scorer.py): it
# ports FinQA's official evaluation protocol (sympy-based symbolic Program
# Accuracy, table-row-lookup-aware Execution Accuracy) instead of a
# hand-rolled parser, and correctly executes table_*(row_name, none) calls by
# looking up the named row in the raw table -- which the previous in-notebook
# parser could not do at all (it treated table_* arguments as raw numbers).

In [ ]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print(summary)

In [ ]:
results_path = BATCH_DIR / "scored_results.csv"
summary_path = BATCH_DIR / "summary.json"

df_scored.to_csv(results_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "shot": "0-shot", "reasoning_effort": REASONING_EFFORT, "api": "batch", **summary}, f, ensure_ascii=False, indent=2)

print(f"Saved per-sample results to {results_path}")
print(f"Saved summary to {summary_path}")